In [0]:
ruta_empleados = "/Volumes/electrocasa_dev/bronze/landing/empleados/empleados_rrhh.csv"

empleados = (
    spark.read
        .option("header", "true")
        .csv(ruta_empleados)
)

empleados.printSchema()
display(empleados.limit(10))

In [0]:
empleados.createOrReplaceTempView("empleados_tmp")

In [0]:
%sql

SELECT
    COUNT(*) AS total_registros,
    COUNT(DISTINCT id_empleado) AS empleados_unicos,

    SUM(CASE
        WHEN dni IS NULL OR TRIM(dni) = ''
        THEN 1 ELSE 0
    END) AS dni_faltante,

    SUM(CASE
        WHEN fecha_evento IS NULL OR TRIM(fecha_evento) = ''
        THEN 1 ELSE 0
    END) AS fecha_evento_faltante,

    SUM(CASE
        WHEN salario IS NULL OR TRIM(salario) = ''
        THEN 1 ELSE 0
    END) AS salario_faltante

FROM empleados_tmp;

In [0]:
%sql

SELECT
    tipo_evento,
    COUNT(*) AS cantidad
FROM empleados_tmp
GROUP BY tipo_evento
ORDER BY tipo_evento;

In [0]:
%sql

SELECT
    cantidad_eventos,
    COUNT(*) AS cantidad_empleados
FROM (
    SELECT
        id_empleado,
        COUNT(*) AS cantidad_eventos
    FROM empleados_tmp
    GROUP BY id_empleado
)
GROUP BY cantidad_eventos
ORDER BY cantidad_eventos;

In [0]:
%sql

SELECT
    COUNT(*) AS pares_empleado_fecha_repetidos
FROM (
    SELECT
        id_empleado,
        fecha_evento,
        COUNT(*) AS cantidad
    FROM empleados_tmp
    WHERE fecha_evento IS NOT NULL
      AND TRIM(fecha_evento) <> ''
    GROUP BY id_empleado, fecha_evento
    HAVING COUNT(*) > 1
);

In [0]:
%sql

SELECT e.*
FROM empleados_tmp e
INNER JOIN (
    SELECT
        id_empleado,
        fecha_evento
    FROM empleados_tmp
    WHERE fecha_evento IS NOT NULL
      AND TRIM(fecha_evento) <> ''
    GROUP BY id_empleado, fecha_evento
    HAVING COUNT(*) > 1
) d
ON e.id_empleado = d.id_empleado
AND e.fecha_evento = d.fecha_evento
ORDER BY e.id_empleado, e.fecha_evento, e.tipo_evento;

In [0]:
%sql

SELECT
    id_empleado,
    fecha_evento,
    lower(
        regexp_replace(
            trim(tipo_evento),
            ' ',
            '_'
        )
    ) AS tipo_evento_normalizado,
    COUNT(*) AS cantidad
FROM empleados_tmp
WHERE fecha_evento IS NOT NULL
  AND TRIM(fecha_evento) <> ''
GROUP BY
    id_empleado,
    fecha_evento,
    lower(
        regexp_replace(
            trim(tipo_evento),
            ' ',
            '_'
        )
    )
HAVING COUNT(*) > 1
ORDER BY id_empleado;

In [0]:
%sql

SELECT
    COUNT(*) AS total_registros,
    COUNT(DISTINCT id_empleado) AS empleados_unicos,
    COUNT(DISTINCT archivo_origen) AS archivos_origen,
    SUM(CASE
        WHEN _rescued_data IS NOT NULL
        THEN 1 ELSE 0
    END) AS registros_rescatados
FROM electrocasa_dev.bronze.empleados;

In [0]:
%sql

SELECT
    id_empleado,
    tipo_evento,
    fecha_evento,
    fec_ingesta,
    archivo_origen,
    id_lote,
    _rescued_data
FROM electrocasa_dev.bronze.empleados
LIMIT 10;

In [0]:
%sql

SELECT
    dni,
    COUNT(DISTINCT id_empleado) AS empleados_distintos
FROM electrocasa_dev.bronze.empleados
WHERE dni IS NOT NULL
  AND TRIM(dni) <> ''
GROUP BY dni
HAVING COUNT(DISTINCT id_empleado) > 1
ORDER BY empleados_distintos DESC, dni;

In [0]:
%sql

SELECT
    dni,
    id_empleado,
    nombre,
    email,
    sucursal_id,
    cargo,
    tipo_evento,
    fecha_evento
FROM electrocasa_dev.bronze.empleados
WHERE dni IN (
    SELECT dni
    FROM electrocasa_dev.bronze.empleados
    WHERE dni IS NOT NULL
      AND TRIM(dni) <> ''
    GROUP BY dni
    HAVING COUNT(DISTINCT id_empleado) > 1
)
ORDER BY dni, id_empleado, fecha_evento;

In [0]:
%sql

SELECT
    id_empleado,
    COUNT(DISTINCT dni) AS dni_distintos
FROM electrocasa_dev.bronze.empleados
WHERE dni IS NOT NULL
  AND TRIM(dni) <> ''
GROUP BY id_empleado
HAVING COUNT(DISTINCT dni) > 1
ORDER BY dni_distintos DESC, id_empleado;

In [0]:
%sql

SELECT
    id_empleado,
    nombre,
    dni,
    tipo_evento,
    fecha_evento,
    sucursal_id,
    salario
FROM electrocasa_dev.bronze.empleados
WHERE id_empleado IN (
    SELECT id_empleado
    FROM electrocasa_dev.bronze.empleados
    WHERE dni IS NOT NULL
      AND TRIM(dni) <> ''
    GROUP BY id_empleado
    HAVING COUNT(DISTINCT dni) > 1
)
ORDER BY id_empleado, fecha_evento;

In [0]:
%sql

WITH altas AS (
    SELECT
        id_empleado,
        dni
    FROM electrocasa_dev.bronze.empleados
    WHERE lower(
        regexp_replace(
            trim(tipo_evento),
            ' ',
            '_'
        )
    ) = 'alta'
),

dni_duplicado_alta AS (
    SELECT dni
    FROM altas
    WHERE dni IS NOT NULL
      AND TRIM(dni) <> ''
    GROUP BY dni
    HAVING COUNT(DISTINCT id_empleado) > 1
)

SELECT
    COUNT(*) AS eventos_alta,
    COUNT(DISTINCT id_empleado) AS empleados_con_alta,

    SUM(CASE
        WHEN dni IS NULL OR TRIM(dni) = ''
        THEN 1 ELSE 0
    END) AS alta_dni_faltante,

    COUNT(DISTINCT CASE
        WHEN dni IS NOT NULL AND TRIM(dni) <> ''
        THEN dni
    END) AS dni_unicos_en_alta,

    (SELECT COUNT(*) FROM dni_duplicado_alta) AS dni_duplicados_en_alta

FROM altas;

In [0]:
%sql

WITH fechas_repetidas AS (
    SELECT
        id_empleado,
        fecha_evento
    FROM electrocasa_dev.bronze.empleados
    WHERE fecha_evento IS NOT NULL
      AND TRIM(fecha_evento) <> ''
    GROUP BY id_empleado, fecha_evento
    HAVING COUNT(*) > 1
)

SELECT
    COUNT(*) AS total_eventos,

    SUM(CASE
        WHEN e.fecha_evento IS NULL
          OR TRIM(e.fecha_evento) = ''
        THEN 1 ELSE 0
    END) AS sin_fecha,

    SUM(CASE
        WHEN f.id_empleado IS NOT NULL
        THEN 1 ELSE 0
    END) AS fecha_ambigua,

    SUM(CASE
        WHEN e.fecha_evento IS NULL
          OR TRIM(e.fecha_evento) = ''
          OR f.id_empleado IS NOT NULL
        THEN 1 ELSE 0
    END) AS eventos_no_historizables

FROM electrocasa_dev.bronze.empleados e
LEFT JOIN fechas_repetidas f
    ON e.id_empleado = f.id_empleado
   AND e.fecha_evento = f.fecha_evento;

In [0]:
%sql

SELECT
    id_empleado,
    nombre,
    salario,
    tipo_evento,
    fecha_evento
FROM electrocasa_dev.bronze.empleados
WHERE salario IS NOT NULL
ORDER BY CAST(salario AS DOUBLE) DESC
LIMIT 15;

In [0]:
%sql

SELECT
    MIN(CAST(salario AS DOUBLE)) AS salario_min,
    AVG(CAST(salario AS DOUBLE)) AS salario_promedio,
    percentile_approx(CAST(salario AS DOUBLE), 0.50) AS p50,
    percentile_approx(CAST(salario AS DOUBLE), 0.95) AS p95,
    percentile_approx(CAST(salario AS DOUBLE), 0.99) AS p99,
    MAX(CAST(salario AS DOUBLE)) AS salario_max
FROM electrocasa_dev.bronze.empleados;

In [0]:
%sql

SELECT
    COUNT(*) AS eventos_salario_atipico,
    COUNT(DISTINCT id_empleado) AS empleados_afectados,
    MIN(CAST(salario AS DOUBLE)) AS minimo_atipico,
    MAX(CAST(salario AS DOUBLE)) AS maximo_atipico
FROM electrocasa_dev.bronze.empleados
WHERE CAST(salario AS DOUBLE) > 10000;

In [0]:
%sql

SELECT
    (SELECT COUNT(*)
     FROM electrocasa_dev.silver.empleados_eventos) AS eventos_validos,

    (SELECT COUNT(*)
     FROM electrocasa_dev.silver.empleados_cuarentena) AS eventos_cuarentena,

    (SELECT COUNT(*)
     FROM electrocasa_dev.silver.empleados_eventos
     WHERE salario_atipico = true) AS salarios_atipicos_validos,

    (SELECT COUNT(*)
     FROM electrocasa_dev.silver.empleados_cuarentena
     WHERE motivo_rechazo = 'fecha_evento_faltante') AS sin_fecha,

    (SELECT COUNT(*)
     FROM electrocasa_dev.silver.empleados_cuarentena
     WHERE motivo_rechazo = 'fecha_evento_ambigua') AS fecha_ambigua;

In [0]:
%sql

SELECT
    COUNT(*) AS total_versiones,
    COUNT(DISTINCT id_empleado) AS empleados_historizados,

    SUM(CASE
        WHEN __END_AT IS NULL
        THEN 1 ELSE 0
    END) AS versiones_vigentes,

    SUM(CASE
        WHEN __END_AT IS NOT NULL
        THEN 1 ELSE 0
    END) AS versiones_historicas,

    MIN(__START_AT) AS primera_vigencia,
    MAX(__START_AT) AS ultima_vigencia

FROM electrocasa_dev.silver.empleados_historial;

In [0]:
%sql

SELECT
    id_empleado,
    nombre,
    dni,
    salario,
    sucursal_id,
    cargo,
    salario_atipico,
    __START_AT,
    __END_AT
FROM electrocasa_dev.silver.empleados_historial
WHERE id_empleado = 'E01093'
ORDER BY __START_AT;

In [0]:
%sql

SELECT DISTINCT
    b.id_empleado
FROM electrocasa_dev.bronze.empleados b
LEFT ANTI JOIN (
    SELECT DISTINCT id_empleado
    FROM electrocasa_dev.silver.empleados_historial
) h
ON b.id_empleado = h.id_empleado
ORDER BY b.id_empleado;

In [0]:
%sql

SELECT
    id_empleado,
    nombre,
    dni,
    tipo_evento,
    fecha_evento,
    sucursal_id,
    salario
FROM electrocasa_dev.bronze.empleados
WHERE id_empleado IN (
    'E00390',
    'E00760',
    'E01324',
    'E01837',
    'E01914'
)
ORDER BY id_empleado, fecha_evento;

In [0]:
%sql

SELECT
    id_empleado,
    nombre,
    dni,
    salario,
    sucursal_id,
    cargo,
    __START_AT,
    __END_AT
FROM electrocasa_dev.silver.empleados_historial
WHERE id_empleado = 'E01924'
ORDER BY __START_AT;